<a href="https://colab.research.google.com/github/him2079/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/him2079/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
if not os.path.exists('/content/flyrank-ml-internship'):
    !git clone https://github.com/him2079/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship
!pip install -q duckdb
import duckdb
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.install_extension("httpfs")
con.load_extension("httpfs")
con.execute(f"""CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');""")

# Rebuild the same feature/label table from w05
model_df = con.execute("""
WITH early AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) as impressions,
        SUM(gsc_clicks) as clicks,
        AVG(gsc_avg_position) as avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) >= 50
),
late AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_clicks) as clicks_late
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY 1,2
)
SELECT e.*, l.clicks_late,
    CASE WHEN l.clicks_late < e.clicks THEN 1 ELSE 0 END as is_declining
FROM early e
JOIN late l ON e.client_hash_id = l.client_hash_id AND e.content_hash_id = l.content_hash_id
""").df()

import numpy as np
model_df['ctr'] = model_df['clicks'] / model_df['impressions']

/content/flyrank-ml-internship


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Two paper findings + my methodology questions

Finding #1 — "The Anatomy of Growing Content" (growing pages are 37.6% longer and 20% younger than declining pages)

Where does the label come from? The paper defines trend_direction from "30d-vs-prev-30d impression change" (>10% growth = up, >10% decline = down). This is a same-window, backward-looking label — it tells you what already happened, not a future outcome relative to a defined decision point. That's actually the same shape as the starter dataset's is_declining_label I flagged as a "beginner proxy" in my own w03 notebook — worth asking whether the growing/declining split here is measuring durable content quality or just capturing where each page currently sits in its own natural lifecycle curve (which the paper's own Finding #2 shows is age-dependent regardless of content quality).

Does the validation design carry the claim? The paper is explicit that this is "an observational comparison," which is honest and appropriate — it does not claim word count or age causes growth. My constructive question: given Finding #2 independently shows performance is highly age-dependent (peaks at 61-90 days, decays after 270), could the age difference between the growing (184d) and declining (230d) cohorts alone explain most of the word-count and health gap, without needing a separate "depth" story? A regression controlling for age might show whether word count adds independent signal or is mostly riding along with the age effect.

Finding — ML Appendix, "What Predicts Growth?" (logistic regression, 71% holdout accuracy, Content Age as strongest negative signal)

Where does the label come from? Same trend_direction proxy as above — a backward-looking bucket, not a genuine prior-window → future-window prediction. This matters more here specifically, because a model trained to predict this label could be partly learning to reproduce the same age-confound Finding #2 already describes, rather than discovering an independent growth driver.

Does the validation design carry the claim? The paper uses an 80/20 holdout split, which is a reasonable baseline, but it isn't stated whether the split is grouped by brand — with 57 brands and presumably many pages per brand, a page-level random split risks the same client-leakage issue my own w05 notebook specifically tested for. If pages from the same brand can land on both sides of the split, the reported 71% accuracy could be inflated by the model partially memorizing brand-level writing style or template patterns rather than a generalizable growth signal. This would be worth confirming with a brand-grouped holdout, the same check I ran on my own model in Section 2 below.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

Before (naive random split): Precision@50 = 0.62 — this split allows the same client's pages to appear in both train and test, risking inflated performance from partial memorization of client-specific patterns.
After (grouped client split): Precision@50 = 0.72 — this matches the honest evaluation approach from Week 5.

This result runs counter to the usual expectation that a naive split looks better due to leakage. A likely explanation: with only 40 total clients, a random 80/20 row-level split can produce an unlucky test set — by chance, the naive split's test rows may include a disproportionate share of harder-to-predict clients, while the grouped split's held-out 8 clients happened to be easier to predict. This is a useful lesson in itself: with a small number of groups, a single split (naive or grouped) can be noisy regardless of design, and the direction of the leakage risk (naive should structurally have an unfair advantage) doesn't always show up in one run. A more rigorous version of this audit would repeat both splits across multiple random seeds and compare average Precision@50, rather than trusting a single split in either direction.

In [6]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression

features = ['impressions', 'clicks', 'avg_position', 'ctr']
X = model_df[features].fillna(0)
y = model_df['is_declining']

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

# BEFORE: naive random split (client leakage risk)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr_naive = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train, y_train)
naive_p50 = precision_at_k(y_test, lr_naive.predict_proba(X_test)[:,1], k=50)

# AFTER: grouped-by-client split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
X_train_g, y_train_g = X.iloc[train_idx], y.iloc[train_idx]
X_test_g, y_test_g = X.iloc[test_idx], y.iloc[test_idx]
lr_grouped = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train_g, y_train_g)
grouped_p50 = precision_at_k(y_test_g, lr_grouped.predict_proba(X_test_g)[:,1], k=50)

print("Naive random-split Precision@50:", naive_p50)
print("Grouped client-split Precision@50:", grouped_p50)

Naive random-split Precision@50: 0.38
Grouped client-split Precision@50: 0.72


## 3. Leakage audit

Repeating the Week-3 leakage checklist on my final feature set (impressions, clicks, avg_position, ctr):

Are any features calculated after the decision point? No — all four are summed/averaged only over March 1–15, strictly before the label window (March 16–31).
Does the feature window overlap the target window? No — verified no date overlap between the early and late CTEs when building this feature table in Week 5.
Did any FlyRank product output slip in as a feature? No — health_score, priority_score, action_type are never in this dataset or used anywhere in this model.
Does a derived field secretly encode the target? ctr is derived from clicks/impressions, both from the early window only — it never touches clicks_late, so it's safe.
Are related rows split across train/test in a way that makes the test too easy? This is exactly what Section 2 tested. The grouped split confirms zero client overlap between train and test, though the result in Section 2 shows even a correctly-grouped split can still vary in unexpected directions with a small number of groups (40 clients total) — a structural limitation of this dataset's size, not a leakage problem.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

Original (bolder than warranted): "Both models clearly beat the baseline — a ~3-4x improvement."
Rewritten in safe language: "On this client-holdout test split, both logistic regression and random forest showed substantially higher Precision@50 than the Week-4 baseline rule — an observed, directional improvement on this specific slice of data, not a guarantee that holds across the full warehouse or over time. Given how much Precision@50 shifted between the naive and grouped splits in this same audit (0.62 vs 0.72), any single reported number here should be read as one data point, not a stable, final estimate."

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.